In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import os
from collections import defaultdict

# ============================================================
# SECTION 1: Environment — 3-Link Planar Arm
# ============================================================

class PlanarArm:
    """
    3-link planar robot arm.
    State: joint angles [θ1, θ2, θ3]
    Action: joint velocities [dθ1, dθ2, dθ3]
    """
    def __init__(self, link_lengths=(1.0, 0.8, 0.6)):
        self.link_lengths = np.array(link_lengths)
        self.n_joints = len(link_lengths)
        self.dt = 0.05  # timestep

    def forward_kinematics(self, joints):
        """Compute end-effector position from joint angles."""
        x, y = 0.0, 0.0
        angle_sum = 0.0
        positions = [(x, y)]
        for i in range(self.n_joints):
            angle_sum += joints[i]
            x += self.link_lengths[i] * np.cos(angle_sum)
            y += self.link_lengths[i] * np.sin(angle_sum)
            positions.append((x, y))
        return np.array([x, y]), positions

    def jacobian(self, joints):
        """Compute the 2x3 Jacobian."""
        J = np.zeros((2, self.n_joints))
        for i in range(self.n_joints):
            angle_sum = np.sum(joints[:i+1])
            for j in range(i+1):
                # Contribution of joint j to link i
                pass
        # Correct Jacobian computation
        J = np.zeros((2, self.n_joints))
        for j in range(self.n_joints):
            angle_sum = np.sum(joints[:j+1])
            # Joint j affects all links from j onward
            for k in range(j, self.n_joints):
                cumulative_angle = np.sum(joints[:k+1])
                J[0, j] += -self.link_lengths[k] * np.sin(cumulative_angle)
                J[1, j] +=  self.link_lengths[k] * np.cos(cumulative_angle)
        return J

    def step(self, joints, action):
        """Apply joint velocity action, return new joints."""
        new_joints = joints + action * self.dt
        # Wrap to [-pi, pi]
        new_joints = (new_joints + np.pi) % (2 * np.pi) - np.pi
        return new_joints

    def inverse_kinematics(self, target, q_init=None, max_iter=200, tol=1e-3):
        """Damped least squares IK from a given initial config."""
        if q_init is None:
            q_init = np.random.uniform(-np.pi, np.pi, self.n_joints)
        q = q_init.copy()
        damping = 0.1

        for _ in range(max_iter):
            ee, _ = self.forward_kinematics(q)
            error = target - ee
            if np.linalg.norm(error) < tol:
                return q, True
            J = self.jacobian(q)
            # Damped least squares: dq = J^T (J J^T + λ²I)^{-1} e
            JJt = J @ J.T + damping**2 * np.eye(2)
            dq = J.T @ np.linalg.solve(JJt, error)
            q = q + dq * 0.5
            q = (q + np.pi) % (2 * np.pi) - np.pi

        return q, False


# ============================================================
# SECTION 2: Expert Demonstration Generation
# ============================================================

def generate_demonstrations(arm, n_demos=500, n_targets=20, steps_per_demo=40):
    """
    Generate expert demonstrations.

    For each target, solve IK from MULTIPLE random initial configs
    → multiple trajectories to same goal → empirical entropy structure.

    Each demonstration: sequence of (state, action, next_state, goal, timestep)
    """
    demonstrations = []
    targets = []

    # Generate reachable targets
    max_reach = sum(arm.link_lengths) * 0.7  # stay within comfortable range
    min_reach = arm.link_lengths[0] * 0.3

    for _ in range(n_targets):
        angle = np.random.uniform(-np.pi, np.pi)
        radius = np.random.uniform(min_reach, max_reach)
        target = np.array([radius * np.cos(angle), radius * np.sin(angle)])
        targets.append(target)

    demos_per_target = n_demos // n_targets

    for target in targets:
        for _ in range(demos_per_target):
            # Random starting configuration
            q_start = np.random.uniform(-np.pi/2, np.pi/2, arm.n_joints)

            # Solve IK for goal configuration
            q_goal, success = arm.inverse_kinematics(target, q_init=None)
            if not success:
                continue

            # Generate smooth trajectory via linear interpolation in joint space
            # (simple but effective for this proof of concept)
            trajectory = []
            for t in range(steps_per_demo):
                alpha = t / (steps_per_demo - 1)
                # Smooth interpolation with cosine schedule
                alpha_smooth = 0.5 * (1 - np.cos(np.pi * alpha))
                q_t = q_start * (1 - alpha_smooth) + q_goal * alpha_smooth
                q_t = (q_t + np.pi) % (2 * np.pi) - np.pi

                if t < steps_per_demo - 1:
                    alpha_next = (t + 1) / (steps_per_demo - 1)
                    alpha_next_smooth = 0.5 * (1 - np.cos(np.pi * alpha_next))
                    q_next = q_start * (1 - alpha_next_smooth) + q_goal * alpha_next_smooth
                    q_next = (q_next + np.pi) % (2 * np.pi) - np.pi
                    action = (q_next - q_t) / arm.dt
                else:
                    action = np.zeros(arm.n_joints)

                ee_pos, _ = arm.forward_kinematics(q_t)

                trajectory.append({
                    'state': q_t.copy(),           # joint angles
                    'ee_pos': ee_pos.copy(),        # end-effector position
                    'action': action.copy(),        # joint velocities
                    'goal': target.copy(),          # target EE position
                    'q_goal': q_goal.copy(),        # goal joint config
                    'timestep': t,
                    'total_steps': steps_per_demo,
                    'normalized_time': alpha,        # 0→1, proxy for entropy
                })

            demonstrations.append(trajectory)

    print(f"Generated {len(demonstrations)} demonstrations "
          f"to {n_targets} targets")
    return demonstrations, targets


# ============================================================
# SECTION 3: Dataset Preparation
# ============================================================

class EFPDataset(Dataset):
    """
    Dataset for training all three EFP components.
    Each sample: (state, action, next_state, goal, entropy_label)
    """
    def __init__(self, demonstrations):
        self.states = []
        self.actions = []
        self.next_states = []
        self.goals = []
        self.entropy_labels = []
        self.ee_positions = []
        self.timesteps = []

        for traj in demonstrations:
            for i, step in enumerate(traj[:-1]):
                self.states.append(step['state'])
                self.actions.append(step['action'])
                self.next_states.append(traj[i+1]['state'])
                self.goals.append(step['q_goal'])  # goal in joint space
                self.ee_positions.append(step['ee_pos'])

                # Entropy label: proportional to remaining steps
                # More remaining steps → higher entropy
                remaining = step['total_steps'] - step['timestep']
                self.entropy_labels.append(np.log(remaining + 1))

                self.timesteps.append(step['timestep'])

        self.states = torch.FloatTensor(np.array(self.states))
        self.actions = torch.FloatTensor(np.array(self.actions))
        self.next_states = torch.FloatTensor(np.array(self.next_states))
        self.goals = torch.FloatTensor(np.array(self.goals))
        self.entropy_labels = torch.FloatTensor(np.array(self.entropy_labels))
        self.ee_positions = torch.FloatTensor(np.array(self.ee_positions))
        self.timesteps = torch.LongTensor(np.array(self.timesteps))

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return {
            'state': self.states[idx],
            'action': self.actions[idx],
            'next_state': self.next_states[idx],
            'goal': self.goals[idx],
            'entropy_label': self.entropy_labels[idx],
            'ee_pos': self.ee_positions[idx],
            'timestep': self.timesteps[idx],
        }


class PairDataset(Dataset):
    """
    Dataset of state PAIRS from same trajectory for ordering loss.
    (s_early, s_late, goal) → H(s_early) > H(s_late)
    """
    def __init__(self, demonstrations, pairs_per_traj=20):
        self.s_early = []
        self.s_late = []
        self.goals = []

        for traj in demonstrations:
            n = len(traj)
            for _ in range(pairs_per_traj):
                i = np.random.randint(0, n - 2)
                j = np.random.randint(i + 1, n)
                self.s_early.append(traj[i]['state'])
                self.s_late.append(traj[j]['state'])
                self.goals.append(traj[0]['q_goal'])

        self.s_early = torch.FloatTensor(np.array(self.s_early))
        self.s_late = torch.FloatTensor(np.array(self.s_late))
        self.goals = torch.FloatTensor(np.array(self.goals))

    def __len__(self):
        return len(self.s_early)

    def __getitem__(self, idx):
        return self.s_early[idx], self.s_late[idx], self.goals[idx]


# ============================================================
# SECTION 4: Neural Network Components
# ============================================================

class EntropyEstimator(nn.Module):
    """
    H_θ(s, s_g) → scalar entropy estimate.
    Input: [state; goal] concatenated
    Output: non-negative scalar (entropy)

    Trained with ordering loss: H(s_early) > H(s_late) for same trajectory.
    """
    def __init__(self, state_dim=3, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Softplus(),  # ensure non-negative entropy
        )

    def forward(self, state, goal):
        x = torch.cat([state, goal], dim=-1)
        return self.net(x).squeeze(-1)


class DynamicsModel(nn.Module):
    """
    f_φ(s, a) → s'
    Simple forward dynamics predictor.
    """
    def __init__(self, state_dim=3, action_dim=3, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, state_dim),
        )
        # Residual: predict delta
        self.residual = True

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        delta = self.net(x)
        if self.residual:
            return state + delta
        return delta


class EntropicScoreNetwork(nn.Module):
    """
    ε_ψ(a_noisy, s, s_g, k) → action correction direction

    This replaces the noise prediction network in DDPM.
    Instead of predicting added noise, it predicts the direction
    in action space that most reduces state entropy.

    k = diffusion step (integer, embedded)
    """
    def __init__(self, state_dim=3, action_dim=3, hidden_dim=128, max_steps=20):
        super().__init__()
        self.step_embed = nn.Embedding(max_steps, 32)
        self.net = nn.Sequential(
            nn.Linear(action_dim + state_dim * 2 + 32, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, noisy_action, state, goal, step):
        step_emb = self.step_embed(step)
        x = torch.cat([noisy_action, state, goal, step_emb], dim=-1)
        return self.net(x)


class BehavioralCloning(nn.Module):
    """
    Baseline: standard behavioral cloning π_BC(s, s_g) → a
    """
    def __init__(self, state_dim=3, action_dim=3, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, state, goal):
        x = torch.cat([state, goal], dim=-1)
        return self.net(x)


# ============================================================
# SECTION 5: Training
# ============================================================

def train_entropy_estimator(model, pair_dataset, dataset, epochs=200, lr=1e-3):
    """
    Train H_θ with:
      1. Ordering loss: H(s_early, g) > H(s_late, g) + margin
      2. Anchor loss: H(s_goal, s_goal) ≈ 0
      3. Regression loss: H ≈ log(remaining_steps) (soft supervision)
    """
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    pair_loader = DataLoader(pair_dataset, batch_size=256, shuffle=True)
    reg_loader = DataLoader(dataset, batch_size=256, shuffle=True)
    margin = 0.3

    losses_history = []

    for epoch in range(epochs):
        total_loss = 0
        n_batches = 0

        # Ordering loss
        for s_early, s_late, goals in pair_loader:
            h_early = model(s_early, goals)
            h_late = model(s_late, goals)

            # H(early) should be > H(late) by at least margin
            order_loss = torch.relu(h_late - h_early + margin).mean()

            # Anchor: H(goal, goal) = 0
            h_goal = model(goals, goals)
            anchor_loss = (h_goal ** 2).mean()

            loss = order_loss + 0.5 * anchor_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        # Regression loss (soft target)
        for batch in reg_loader:
            h_pred = model(batch['state'], batch['goal'])
            h_target = batch['entropy_label']
            reg_loss = nn.functional.mse_loss(h_pred, h_target)

            optimizer.zero_grad()
            (0.3 * reg_loss).backward()
            optimizer.step()
            total_loss += reg_loss.item()
            n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)
        losses_history.append(avg_loss)
        if (epoch + 1) % 20 == 0:
            print(f"  Entropy Estimator Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    return losses_history


def train_dynamics_model(model, dataset, epochs=200, lr=1e-3):
    """Train f_φ(s, a) → s' with MSE loss."""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)

    losses_history = []

    for epoch in range(epochs):
        total_loss = 0
        n_batches = 0
        for batch in loader:
            s_pred = model(batch['state'], batch['action'])
            loss = nn.functional.mse_loss(s_pred, batch['next_state'])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)
        losses_history.append(avg_loss)
        if (epoch + 1) % 20 == 0:
            print(f"  Dynamics Model Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    return losses_history


def train_score_network(score_net, entropy_net, dynamics_net, dataset,
                        n_diffusion_steps=10, epochs=200, lr=1e-3):
    """
    Train ε_ψ to predict the entropic gradient direction.

    For each (s, a_expert, g):
      1. Add noise to a_expert at random diffusion step k
      2. Compute target = direction toward entropy reduction
         (approximated by: a_expert - a_noisy, scaled)
      3. Train ε_ψ to predict this direction
    """
    optimizer = optim.AdamW(score_net.parameters(), lr=lr, weight_decay=1e-4)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)

    # Noise schedule (linear, like DDPM)
    betas = torch.linspace(0.01, 0.5, n_diffusion_steps)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)

    losses_history = []

    for epoch in range(epochs):
        total_loss = 0
        n_batches = 0

        for batch in loader:
            a_expert = batch['action']
            state = batch['state']
            goal = batch['goal']
            bsz = a_expert.shape[0]

            # Random diffusion step
            k = torch.randint(0, n_diffusion_steps, (bsz,))
            alpha_bar_k = alpha_bars[k].unsqueeze(-1)

            # Add noise
            noise = torch.randn_like(a_expert)
            a_noisy = torch.sqrt(alpha_bar_k) * a_expert + \
                      torch.sqrt(1 - alpha_bar_k) * noise

            # Target: predict the noise (standard DDPM objective)
            # But conceptually, this noise points AWAY from the
            # entropy-minimizing action, so predicting it lets us
            # reverse toward entropy reduction
            predicted_noise = score_net(a_noisy, state, goal, k)

            loss = nn.functional.mse_loss(predicted_noise, noise)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)
        losses_history.append(avg_loss)
        if (epoch + 1) % 20 == 0:
            print(f"  Score Network Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    return losses_history, betas, alpha_bars


def train_bc_baseline(model, dataset, epochs=200, lr=1e-3):
    """Train behavioral cloning baseline."""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)

    losses_history = []

    for epoch in range(epochs):
        total_loss = 0
        n_batches = 0
        for batch in loader:
            a_pred = model(batch['state'], batch['goal'])
            loss = nn.functional.mse_loss(a_pred, batch['action'])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)
        losses_history.append(avg_loss)
        if (epoch + 1) % 20 == 0:
            print(f"  BC Baseline Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    return losses_history


# ============================================================
# SECTION 6: Inference — Entropic Diffusion Sampling
# ============================================================

def entropic_diffusion_sample(score_net, entropy_net, dynamics_net, state, goal,
                              betas, alpha_bars, n_steps=10, guidance_scale=1.0):
    """
    Generate an action via entropic diffusion.

    Key difference from standard diffusion:
      - Noise level is modulated by current state entropy
      - Denoising direction incorporates entropy gradient

    Args:
        state: current joint angles [3]
        goal: target joint angles [3]
        n_steps: diffusion denoising steps
        guidance_scale: how much to weight entropy gradient vs learned score
    """
    state_t = torch.FloatTensor(state).unsqueeze(0)
    goal_t = torch.FloatTensor(goal).unsqueeze(0)

    with torch.no_grad():
        # Estimate current state entropy
        h_current = entropy_net(state_t, goal_t).item()

    # Adaptive noise: scale initial noise by entropy
    # High entropy → more noise → broader search
    # Low entropy → less noise → precise action
    entropy_scale = min(h_current / 3.0, 1.0)  # normalize roughly

    # Start from noise, scaled by entropy
    a_t = torch.randn(1, 3) * entropy_scale

    alphas = 1.0 - betas

    with torch.no_grad():
        for k in reversed(range(n_steps)):
            k_tensor = torch.LongTensor([k])
            alpha_k = alphas[k]
            alpha_bar_k = alpha_bars[k]

            # Predict noise
            pred_noise = score_net(a_t, state_t, goal_t, k_tensor)

            # Standard DDPM reverse step
            a_mean = (1 / torch.sqrt(alpha_k)) * (
                a_t - (betas[k] / torch.sqrt(1 - alpha_bar_k)) * pred_noise
            )

            # Add entropy-guided correction
            if guidance_scale > 0 and k > 0:
                # Estimate entropy gradient numerically
                eps = 0.01
                grad = torch.zeros_like(a_mean)
                for d in range(3):
                    a_plus = a_mean.clone()
                    a_plus[0, d] += eps
                    a_minus = a_mean.clone()
                    a_minus[0, d] -= eps

                    s_plus = dynamics_net(state_t, a_plus)
                    s_minus = dynamics_net(state_t, a_minus)

                    h_plus = entropy_net(s_plus, goal_t)
                    h_minus = entropy_net(s_minus, goal_t)

                    grad[0, d] = (h_plus - h_minus) / (2 * eps)

                # Push toward lower entropy
                a_mean = a_mean - guidance_scale * grad * entropy_scale

            # Add noise (except at last step)
            if k > 0:
                noise = torch.randn_like(a_t) * torch.sqrt(betas[k]) * entropy_scale
                a_t = a_mean + noise
            else:
                a_t = a_mean

    return a_t.squeeze(0).numpy()


# ============================================================
# SECTION 7: Evaluation
# ============================================================

def evaluate_policy(arm, policy_fn, targets, q_starts, max_steps=60):
    """
    Evaluate a policy by rolling out trajectories.
    Returns: success rate, average final distance, trajectory data
    """
    results = []

    for target, q_start in zip(targets, q_starts):
        q = q_start.copy()
        trajectory = [q.copy()]
        ee_traj = []
        entropy_traj = []

        # Get a goal joint config via IK
        q_goal, success = arm.inverse_kinematics(target)
        if not success:
            continue

        for t in range(max_steps):
            ee, _ = arm.forward_kinematics(q)
            ee_traj.append(ee.copy())

            dist = np.linalg.norm(ee - target)
            if dist < 0.05:
                break

            action = policy_fn(q, q_goal)
            action = np.clip(action, -5.0, 5.0)
            q = arm.step(q, action)
            trajectory.append(q.copy())

        final_ee, _ = arm.forward_kinematics(q)
        final_dist = np.linalg.norm(final_ee - target)

        results.append({
            'target': target,
            'final_dist': final_dist,
            'success': final_dist < 0.1,
            'steps': len(trajectory),
            'trajectory': trajectory,
            'ee_trajectory': ee_traj,
        })

    return results


def run_experiment():
    """Main experiment: train EFP and BC, compare."""

    print("=" * 60)
    print("ENTROPIC FLOW POLICY — Proof of Concept Experiment")
    print("=" * 60)

    # --- Setup ---
    arm = PlanarArm(link_lengths=(1.0, 0.8, 0.6))
    np.random.seed(42)
    torch.manual_seed(42)

    # --- Generate demonstrations ---
    print("\n[1] Generating expert demonstrations...")
    demos, train_targets = generate_demonstrations(
        arm, n_demos=600, n_targets=30, steps_per_demo=40
    )

    # --- Prepare datasets ---
    print("\n[2] Preparing datasets...")
    dataset = EFPDataset(demos)
    pair_dataset = PairDataset(demos, pairs_per_traj=30)
    print(f"  Transition samples: {len(dataset)}")
    print(f"  Ordering pairs: {len(pair_dataset)}")

    # --- Train components ---
    entropy_net = EntropyEstimator(state_dim=3, hidden_dim=128)
    dynamics_net = DynamicsModel(state_dim=3, action_dim=3, hidden_dim=128)
    score_net = EntropicScoreNetwork(state_dim=3, action_dim=3, hidden_dim=128)
    bc_net = BehavioralCloning(state_dim=3, action_dim=3, hidden_dim=128)

    print("\n[3] Training Entropy Estimator...")
    h_losses = train_entropy_estimator(entropy_net, pair_dataset, dataset, epochs=200)

    print("\n[4] Training Dynamics Model...")
    d_losses = train_dynamics_model(dynamics_net, dataset, epochs=200)

    print("\n[5] Training Entropic Score Network...")
    s_losses, betas, alpha_bars = train_score_network(
        score_net, entropy_net, dynamics_net, dataset, epochs=200
    )

    print("\n[6] Training Behavioral Cloning Baseline...")
    bc_losses = train_bc_baseline(bc_net, dataset, epochs=200)

    # --- Evaluation ---
    print("\n[7] Evaluating policies...")

    # Generate test scenarios
    n_test = 50
    test_targets = []
    test_q_starts = []
    max_reach = sum(arm.link_lengths) * 0.6
    min_reach = arm.link_lengths[0] * 0.4
    np.random.seed(999)
    for _ in range(n_test):
        angle = np.random.uniform(-np.pi, np.pi)
        radius = np.random.uniform(min_reach, max_reach)
        test_targets.append(np.array([radius * np.cos(angle), radius * np.sin(angle)]))
        test_q_starts.append(np.random.uniform(-np.pi/2, np.pi/2, 3))

    # EFP policy
    entropy_net.eval()
    dynamics_net.eval()
    score_net.eval()

    def efp_policy(q, q_goal):
        return entropic_diffusion_sample(
            score_net, entropy_net, dynamics_net, q, q_goal,
            betas, alpha_bars, n_steps=10, guidance_scale=0.5
        )

    # BC policy
    bc_net.eval()
    def bc_policy(q, q_goal):
        with torch.no_grad():
            s = torch.FloatTensor(q).unsqueeze(0)
            g = torch.FloatTensor(q_goal).unsqueeze(0)
            return bc_net(s, g).squeeze(0).numpy()

    print("\n  Evaluating EFP...")
    efp_results = evaluate_policy(arm, efp_policy, test_targets, test_q_starts)

    print("  Evaluating BC...")
    bc_results = evaluate_policy(arm, bc_policy, test_targets, test_q_starts)

    # --- Metrics ---
    efp_success = np.mean([r['success'] for r in efp_results])
    bc_success = np.mean([r['success'] for r in bc_results])
    efp_dist = np.mean([r['final_dist'] for r in efp_results])
    bc_dist = np.mean([r['final_dist'] for r in bc_results])
    efp_steps = np.mean([r['steps'] for r in efp_results])
    bc_steps = np.mean([r['steps'] for r in bc_results])

    print("\n" + "=" * 60)
    print("RESULTS")
    print("=" * 60)
    print(f"{'Metric':<25} {'EFP':>10} {'BC':>10}")
    print("-" * 45)
    print(f"{'Success Rate':<25} {efp_success:>10.1%} {bc_success:>10.1%}")
    print(f"{'Avg Final Distance':<25} {efp_dist:>10.4f} {bc_dist:>10.4f}")
    print(f"{'Avg Steps':<25} {efp_steps:>10.1f} {bc_steps:>10.1f}")

    # --- Save results for visualization ---
    results_data = {
        'efp_results': efp_results,
        'bc_results': bc_results,
        'h_losses': h_losses,
        'd_losses': d_losses,
        's_losses': s_losses,
        'bc_losses': bc_losses,
        'efp_success': efp_success,
        'bc_success': bc_success,
        'efp_dist': efp_dist,
        'bc_dist': bc_dist,
        'arm': arm,
        'entropy_net': entropy_net,
        'dynamics_net': dynamics_net,
        'test_targets': test_targets,
    }

    return results_data


# ============================================================
# SECTION 8: Visualization
# ============================================================

def create_visualizations(results_data):
    """Generate comprehensive visualization plots."""

    fig = plt.figure(figsize=(20, 16))
    fig.suptitle('Entropic Flow Policy — Experimental Results',
                 fontsize=16, fontweight='bold', y=0.98)

    # 1. Training losses
    ax1 = fig.add_subplot(2, 3, 1)
    ax1.plot(results_data['h_losses'], label='Entropy Est.', color='#2196F3', linewidth=1.5)
    ax1.plot(results_data['bc_losses'], label='BC Baseline', color='#FF5722', linewidth=1.5)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.legend()
    ax1.set_yscale('log')
    ax1.grid(True, alpha=0.3)

    # 2. Score network loss
    ax2 = fig.add_subplot(2, 3, 2)
    ax2.plot(results_data['s_losses'], color='#9C27B0', linewidth=1.5)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Entropic Score Network Loss')
    ax2.grid(True, alpha=0.3)

    # 3. Success rate comparison
    ax3 = fig.add_subplot(2, 3, 3)
    methods = ['EFP\n(Ours)', 'Behavioral\nCloning']
    rates = [results_data['efp_success'], results_data['bc_success']]
    colors = ['#2196F3', '#FF5722']
    bars = ax3.bar(methods, rates, color=colors, width=0.5, edgecolor='black', linewidth=0.5)
    ax3.set_ylabel('Success Rate')
    ax3.set_title('Success Rate Comparison')
    ax3.set_ylim(0, 1.1)
    for bar, rate in zip(bars, rates):
        ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{rate:.1%}', ha='center', va='bottom', fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')

    # 4. Final distance distribution
    ax4 = fig.add_subplot(2, 3, 4)
    efp_dists = [r['final_dist'] for r in results_data['efp_results']]
    bc_dists = [r['final_dist'] for r in results_data['bc_results']]
    ax4.hist(efp_dists, bins=20, alpha=0.6, label='EFP', color='#2196F3', edgecolor='black', linewidth=0.5)
    ax4.hist(bc_dists, bins=20, alpha=0.6, label='BC', color='#FF5722', edgecolor='black', linewidth=0.5)
    ax4.set_xlabel('Final EE Distance to Target')
    ax4.set_ylabel('Count')
    ax4.set_title('Final Distance Distribution')
    ax4.axvline(x=0.1, color='green', linestyle='--', label='Success threshold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    # 5. Example EE trajectories
    ax5 = fig.add_subplot(2, 3, 5)
    n_show = min(8, len(results_data['efp_results']))
    for i in range(n_show):
        if results_data['efp_results'][i]['ee_trajectory']:
            ee_traj = np.array(results_data['efp_results'][i]['ee_trajectory'])
            ax5.plot(ee_traj[:, 0], ee_traj[:, 1], 'b-', alpha=0.4, linewidth=1)
        if results_data['bc_results'][i]['ee_trajectory']:
            ee_traj = np.array(results_data['bc_results'][i]['ee_trajectory'])
            ax5.plot(ee_traj[:, 0], ee_traj[:, 1], 'r-', alpha=0.4, linewidth=1)
    # Plot targets
    for i in range(n_show):
        t = results_data['efp_results'][i]['target']
        ax5.plot(t[0], t[1], 'g*', markersize=10)
    ax5.plot([], [], 'b-', label='EFP')
    ax5.plot([], [], 'r-', label='BC')
    ax5.plot([], [], 'g*', label='Targets')
    ax5.set_xlabel('X')
    ax5.set_ylabel('Y')
    ax5.set_title('End-Effector Trajectories')
    ax5.legend()
    ax5.set_aspect('equal')
    ax5.grid(True, alpha=0.3)

    # 6. Entropy landscape visualization
    ax6 = fig.add_subplot(2, 3, 6)
    entropy_net = results_data['entropy_net']
    entropy_net.eval()

    # Fix θ3=0, vary θ1 and θ2, compute entropy to a fixed goal
    arm = results_data['arm']
    target = results_data['test_targets'][0]
    q_goal_viz, _ = arm.inverse_kinematics(target)

    theta1_range = np.linspace(-np.pi, np.pi, 50)
    theta2_range = np.linspace(-np.pi, np.pi, 50)
    H_map = np.zeros((50, 50))

    with torch.no_grad():
        for i, t1 in enumerate(theta1_range):
            for j, t2 in enumerate(theta2_range):
                s = torch.FloatTensor([[t1, t2, 0.0]])
                g = torch.FloatTensor([q_goal_viz])
                H_map[j, i] = entropy_net(s, g).item()

    im = ax6.contourf(theta1_range, theta2_range, H_map, levels=30, cmap='viridis_r')
    ax6.plot(q_goal_viz[0], q_goal_viz[1], 'r*', markersize=15, label='Goal')
    ax6.set_xlabel('θ₁')
    ax6.set_ylabel('θ₂')
    ax6.set_title('Learned Entropy Landscape (θ₃=0)')
    ax6.legend()
    plt.colorbar(im, ax=ax6, label='H(s, s_g)')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('efp_results.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\nVisualization saved to efp_results.png")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    results = run_experiment()
    create_visualizations(results)
    print("\nExperiment complete!")


ENTROPIC FLOW POLICY — Proof of Concept Experiment

[1] Generating expert demonstrations...
Generated 600 demonstrations to 30 targets

[2] Preparing datasets...
  Transition samples: 23400
  Ordering pairs: 18000

[3] Training Entropy Estimator...
  Entropy Estimator Epoch 20/200, Loss: 0.3210
  Entropy Estimator Epoch 40/200, Loss: 0.2267
  Entropy Estimator Epoch 60/200, Loss: 0.1989
  Entropy Estimator Epoch 80/200, Loss: 0.1873
  Entropy Estimator Epoch 100/200, Loss: 0.1747
  Entropy Estimator Epoch 120/200, Loss: 0.1545
  Entropy Estimator Epoch 140/200, Loss: 0.1661
  Entropy Estimator Epoch 160/200, Loss: 0.1658
  Entropy Estimator Epoch 180/200, Loss: 0.1633
  Entropy Estimator Epoch 200/200, Loss: 0.1661

[4] Training Dynamics Model...
  Dynamics Model Epoch 20/200, Loss: 0.0000
  Dynamics Model Epoch 40/200, Loss: 0.0000
  Dynamics Model Epoch 60/200, Loss: 0.0000
  Dynamics Model Epoch 80/200, Loss: 0.0000
  Dynamics Model Epoch 100/200, Loss: 0.0000
  Dynamics Model Epoch

In [ ]:
"""
Entropic Flow Policy — Hard Experiment v2
==========================================
Fixed: collision-free expert demos, wider wall gap for viable multi-modality,
       no wall collision in step() (open field with cost zones instead).

Approach: Remove hard wall collision. Instead use a SOFT obstacle field:
- Two rectangular obstacles creating a corridor effect
- Multiple paths around them (left/right/between)
- This gives genuine multi-modality without collision blocking issues
"""

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ============================================================
# ENVIRONMENT: 2D Navigation with Two Obstacles
# ============================================================

class ObstacleNavEnv:
    """
    2D point navigation with two rectangular obstacles.

    Layout:
        Workspace: [-3, 3] x [-3, 3]
        Obstacle 1: rectangle at x ∈ [-0.3, 0.3], y ∈ [0.3, 2.5]
        Obstacle 2: rectangle at x ∈ [-0.3, 0.3], y ∈ [-2.5, -0.3]
        Gap between them: y ∈ [-0.3, 0.3]

    This creates THREE viable paths:
        - UPPER: go above obstacle 1
        - LOWER: go below obstacle 2
        - MIDDLE: go through the gap between them

    Start region: x ∈ [-2.5, -1.5]
    Goal: (2.0, 0.0)
    """
    def __init__(self):
        self.dt = 0.1
        self.max_speed = 1.5
        self.goal = np.array([2.0, 0.0])

        # Obstacles (x_min, x_max, y_min, y_max)
        self.obstacles = [
            (-0.3, 0.3, 0.4, 2.5),   # upper block
            (-0.3, 0.3, -2.5, -0.4),  # lower block
        ]

    def in_obstacle(self, pos):
        x, y = pos
        for xmin, xmax, ymin, ymax in self.obstacles:
            if xmin <= x <= xmax and ymin <= y <= ymax:
                return True
        return False

    def step(self, pos, action):
        action = np.clip(action, -self.max_speed, self.max_speed)
        new_pos = pos + action * self.dt
        new_pos[0] = np.clip(new_pos[0], -3, 3)
        new_pos[1] = np.clip(new_pos[1], -3, 3)

        # Hard collision: stay put if in obstacle
        if self.in_obstacle(new_pos):
            # Try to slide along obstacle boundary
            test_x = np.array([new_pos[0], pos[1]])
            test_y = np.array([pos[0], new_pos[1]])
            if not self.in_obstacle(test_x):
                return test_x
            elif not self.in_obstacle(test_y):
                return test_y
            return pos.copy()
        return new_pos

    def reached_goal(self, pos, threshold=0.2):
        return np.linalg.norm(pos - self.goal) < threshold


# ============================================================
# EXPERT DEMONSTRATIONS via waypoint paths
# ============================================================

def generate_demos(env, n_demos=1000, n_points=50):
    demos = []

    for i in range(n_demos):
        start = np.array([
            np.random.uniform(-2.5, -1.5),
            np.random.uniform(-2.0, 2.0),
        ])

        goal = env.goal.copy()

        # Choose mode
        r = np.random.rand()
        if r < 0.35:
            mode = 'upper'
        elif r < 0.70:
            mode = 'lower'
        else:
            mode = 'middle'

        # Generate waypoints that are GUARANTEED collision-free
        if mode == 'upper':
            apex_y = np.random.uniform(2.6, 2.9)
            waypoints = [
                start.copy(),
                np.array([start[0] * 0.5 - 0.3, apex_y * 0.7 + start[1] * 0.3]),
                np.array([-0.3, apex_y]),
                np.array([0.3, apex_y]),
                np.array([0.8, apex_y * 0.5 + goal[1] * 0.5]),
                np.array([1.3, goal[1] + (apex_y - goal[1]) * 0.2]),
                goal.copy(),
            ]
        elif mode == 'lower':
            apex_y = np.random.uniform(-2.9, -2.6)
            waypoints = [
                start.copy(),
                np.array([start[0] * 0.5 - 0.3, apex_y * 0.7 + start[1] * 0.3]),
                np.array([-0.3, apex_y]),
                np.array([0.3, apex_y]),
                np.array([0.8, apex_y * 0.5 + goal[1] * 0.5]),
                np.array([1.3, goal[1] + (apex_y - goal[1]) * 0.2]),
                goal.copy(),
            ]
        else:  # middle - through the gap
            gap_y = np.random.uniform(-0.2, 0.2)
            waypoints = [
                start.copy(),
                np.array([-0.8, gap_y * 0.5 + start[1] * 0.5]),
                np.array([-0.4, gap_y]),
                np.array([0.0, gap_y]),
                np.array([0.4, gap_y]),
                np.array([1.0, gap_y * 0.3 + goal[1] * 0.7]),
                goal.copy(),
            ]

        # Smooth interpolation
        path = []
        segs = len(waypoints) - 1
        pts_per = n_points // segs
        for seg in range(segs):
            p1, p2 = waypoints[seg], waypoints[seg + 1]
            for t in range(pts_per):
                alpha = 0.5 * (1 - np.cos(np.pi * t / pts_per))
                path.append(p1 * (1 - alpha) + p2 * alpha)
        path.append(goal.copy())

        # Add small noise
        path = np.array(path)
        noise = np.random.randn(*path.shape) * 0.04
        noise[0] = 0; noise[-1] = 0
        path += noise

        # Verify collision-free
        has_collision = any(env.in_obstacle(p) for p in path)
        if has_collision:
            continue

        # Build trajectory
        trajectory = []
        for t in range(len(path) - 1):
            action = (path[t+1] - path[t]) / env.dt
            trajectory.append({
                'state': path[t].copy(),
                'action': action.copy(),
                'next_state': path[t+1].copy(),
                'goal': goal.copy(),
                'timestep': t,
                'total_steps': len(path) - 1,
                'mode': mode,
            })
        demos.append(trajectory)

    modes = [d[0]['mode'] for d in demos]
    print(f"Generated {len(demos)} collision-free demonstrations")
    print(f"  Upper: {modes.count('upper')}, Lower: {modes.count('lower')}, Middle: {modes.count('middle')}")
    return demos


# ============================================================
# DATASETS
# ============================================================

class NavDataset(Dataset):
    def __init__(self, demonstrations):
        states, actions, next_states, goals, entropy_labels = [], [], [], [], []
        for traj in demonstrations:
            for step in traj:
                states.append(step['state'])
                actions.append(step['action'])
                next_states.append(step['next_state'])
                goals.append(step['goal'])
                remaining = step['total_steps'] - step['timestep']
                entropy_labels.append(np.log(remaining + 1))
        self.states = torch.FloatTensor(np.array(states))
        self.actions = torch.FloatTensor(np.array(actions))
        self.next_states = torch.FloatTensor(np.array(next_states))
        self.goals = torch.FloatTensor(np.array(goals))
        self.entropy_labels = torch.FloatTensor(np.array(entropy_labels))

    def __len__(self): return len(self.states)
    def __getitem__(self, idx):
        return {'state': self.states[idx], 'action': self.actions[idx],
                'next_state': self.next_states[idx], 'goal': self.goals[idx],
                'entropy_label': self.entropy_labels[idx]}


class PairDataset(Dataset):
    def __init__(self, demos, pairs_per_traj=20):
        s_e, s_l, gs = [], [], []
        for traj in demos:
            n = len(traj)
            for _ in range(pairs_per_traj):
                i = np.random.randint(0, n-2)
                j = np.random.randint(i+1, n)
                s_e.append(traj[i]['state']); s_l.append(traj[j]['state'])
                gs.append(traj[0]['goal'])
        self.s_e = torch.FloatTensor(np.array(s_e))
        self.s_l = torch.FloatTensor(np.array(s_l))
        self.gs = torch.FloatTensor(np.array(gs))
    def __len__(self): return len(self.s_e)
    def __getitem__(self, idx): return self.s_e[idx], self.s_l[idx], self.gs[idx]


# ============================================================
# NETWORKS
# ============================================================

class EntropyNet(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h//2), nn.GELU(),
            nn.Linear(h//2, 1), nn.Softplus())
    def forward(self, s, g):
        return self.net(torch.cat([s, g], -1)).squeeze(-1)

class DynNet(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.GELU(), nn.Linear(h, 2))
    def forward(self, s, a):
        return s + self.net(torch.cat([s, a], -1))

class ScoreNet(nn.Module):
    def __init__(self, h=128, max_k=20):
        super().__init__()
        self.emb = nn.Embedding(max_k, 16)
        self.net = nn.Sequential(
            nn.Linear(2+2+2+16, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.GELU(), nn.Linear(h, 2))
    def forward(self, a, s, g, k):
        return self.net(torch.cat([a, s, g, self.emb(k)], -1))

class BCNet(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.GELU(), nn.Linear(h, 2))
    def forward(self, s, g):
        return self.net(torch.cat([s, g], -1))

class MDN(nn.Module):
    def __init__(self, h=128, K=3):
        super().__init__()
        self.K = K
        self.backbone = nn.Sequential(
            nn.Linear(4, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU())
        self.pi_head = nn.Linear(h, K)
        self.mu_head = nn.Linear(h, K*2)
        self.sig_head = nn.Linear(h, K*2)
    def forward(self, s, g):
        f = self.backbone(torch.cat([s, g], -1))
        pi = torch.softmax(self.pi_head(f), -1)
        mu = self.mu_head(f).reshape(-1, self.K, 2)
        sig = torch.exp(self.sig_head(f).reshape(-1, self.K, 2).clamp(-5, 2))
        return pi, mu, sig
    def sample(self, s, g):
        pi, mu, sig = self.forward(s, g)
        c = torch.multinomial(pi, 1).squeeze(-1)
        idx = torch.arange(len(c))
        return mu[idx, c] + sig[idx, c] * torch.randn_like(sig[idx, c])


# ============================================================
# TRAINING
# ============================================================

def train_entropy(net, pairs, ds, ep=60):
    opt = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    pl = DataLoader(pairs, batch_size=256, shuffle=True)
    rl = DataLoader(ds, batch_size=256, shuffle=True)
    losses = []
    for e in range(ep):
        tot, n = 0, 0
        for se, sl, g in pl:
            he, hl = net(se, g), net(sl, g)
            loss = torch.relu(hl - he + 0.3).mean() + 0.5*(net(g, g)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += 1
        for b in rl:
            reg = nn.functional.mse_loss(net(b['state'], b['goal']), b['entropy_label'])
            opt.zero_grad(); (0.3*reg).backward(); opt.step()
            tot += reg.item(); n += 1
        losses.append(tot/max(n,1))
        if (e+1) % 20 == 0: print(f"  Entropy {e+1}/{ep}: {losses[-1]:.4f}")
    return losses

def train_dyn(net, ds, ep=40):
    opt = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    ld = DataLoader(ds, batch_size=256, shuffle=True)
    losses = []
    for e in range(ep):
        tot, n = 0, 0
        for b in ld:
            loss = nn.functional.mse_loss(net(b['state'], b['action']), b['next_state'])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += 1
        losses.append(tot/max(n,1))
        if (e+1) % 20 == 0: print(f"  Dynamics {e+1}/{ep}: {losses[-1]:.4f}")
    return losses

def train_score(net, ds, K=10, ep=60):
    opt = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    ld = DataLoader(ds, batch_size=256, shuffle=True)
    betas = torch.linspace(0.01, 0.5, K)
    alphas = 1.0 - betas
    abars = torch.cumprod(alphas, 0)
    losses = []
    for e in range(ep):
        tot, n = 0, 0
        for b in ld:
            a = b['action']; bsz = a.shape[0]
            k = torch.randint(0, K, (bsz,))
            ab = abars[k].unsqueeze(-1)
            noise = torch.randn_like(a)
            an = torch.sqrt(ab)*a + torch.sqrt(1-ab)*noise
            loss = nn.functional.mse_loss(net(an, b['state'], b['goal'], k), noise)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += 1
        losses.append(tot/max(n,1))
        if (e+1) % 20 == 0: print(f"  Score {e+1}/{ep}: {losses[-1]:.4f}")
    return losses, betas, abars

def train_bc(net, ds, ep=60):
    opt = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    ld = DataLoader(ds, batch_size=256, shuffle=True)
    losses = []
    for e in range(ep):
        tot, n = 0, 0
        for b in ld:
            loss = nn.functional.mse_loss(net(b['state'], b['goal']), b['action'])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += 1
        losses.append(tot/max(n,1))
        if (e+1) % 20 == 0: print(f"  BC {e+1}/{ep}: {losses[-1]:.4f}")
    return losses

def train_mdn(net, ds, ep=60):
    opt = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    ld = DataLoader(ds, batch_size=256, shuffle=True)
    losses = []
    for e in range(ep):
        tot, n = 0, 0
        for b in ld:
            pi, mu, sig = net(b['state'], b['goal'])
            a = b['action'].unsqueeze(1)
            lp = -0.5*((a-mu)/sig)**2 - torch.log(sig) - 0.5*np.log(2*np.pi)
            lp = lp.sum(-1) + torch.log(pi+1e-8)
            loss = -torch.logsumexp(lp, -1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += 1
        losses.append(tot/max(n,1))
        if (e+1) % 20 == 0: print(f"  MDN {e+1}/{ep}: {losses[-1]:.4f}")
    return losses


# ============================================================
# INFERENCE
# ============================================================

def efp_sample(score_net, entropy_net, dyn_net, state, goal, betas, abars, K=5, guidance=0.3):
    s = torch.FloatTensor(state).unsqueeze(0)
    g = torch.FloatTensor(goal).unsqueeze(0)
    with torch.no_grad():
        h = entropy_net(s, g).item()
    escale = min(h / 3.0, 1.0)
    a = torch.randn(1, 2) * escale
    al = 1.0 - betas
    with torch.no_grad():
        for k in reversed(range(K)):
            pn = score_net(a, s, g, torch.LongTensor([k]))
            am = (1/torch.sqrt(al[k]))*(a - (betas[k]/torch.sqrt(1-abars[k]))*pn)
            if guidance > 0 and k > 0 and k % 2 == 0:
                eps = 0.02
                sf = dyn_net(s, am)
                hc = entropy_net(sf, g)
                grad = torch.zeros_like(am)
                for d in range(2):
                    ap = am.clone(); ap[0,d] += eps
                    grad[0,d] = (entropy_net(dyn_net(s, ap), g) - hc) / eps
                am = am - guidance * grad * escale
            if k > 0:
                a = am + torch.randn_like(a)*torch.sqrt(betas[k])*escale
            else:
                a = am
    return a.squeeze(0).numpy()


# ============================================================
# EVALUATION
# ============================================================

def rollout(env, pol, start, max_t=80, perturb_t=None):
    pos = start.copy()
    traj = [pos.copy()]
    for t in range(max_t):
        if env.reached_goal(pos): break
        if perturb_t and t == perturb_t:
            pos = pos + np.random.randn(2)*0.4
            pos[0] = np.clip(pos[0], -3, 3)
            pos[1] = np.clip(pos[1], -3, 3)
        a = pol(pos, env.goal)
        a = np.clip(a, -env.max_speed, env.max_speed)
        pos = env.step(pos, a)
        traj.append(pos.copy())
    return {'trajectory': np.array(traj),
            'final_dist': np.linalg.norm(pos - env.goal),
            'success': env.reached_goal(pos), 'steps': len(traj)}


# ============================================================
# MAIN
# ============================================================

def main():
    print("="*60)
    print("ENTROPIC FLOW POLICY — Multi-Modal Obstacle Navigation v2")
    print("="*60)

    np.random.seed(42); torch.manual_seed(42)
    env = ObstacleNavEnv()

    print("\n[1] Generating demonstrations...")
    demos = generate_demos(env, n_demos=1000, n_points=48)

    print("\n[2] Datasets...")
    ds = NavDataset(demos)
    pairs = PairDataset(demos, 20)
    print(f"  {len(ds)} transitions, {len(pairs)} pairs")

    H = 128
    enet = EntropyNet(H); dnet = DynNet(H); snet = ScoreNet(H)
    bcnet = BCNet(H); mdn = MDN(H, 3)

    print("\n[3] Training entropy estimator...")
    hl = train_entropy(enet, pairs, ds, 100)
    print("\n[4] Training dynamics...")
    dl = train_dyn(dnet, ds, 100)
    print("\n[5] Training score network...")
    sl, betas, abars = train_score(snet, ds, 10, 100)
    print("\n[6] Training BC...")
    bl = train_bc(bcnet, ds, 100)
    print("\n[7] Training MDN-BC...")
    ml = train_mdn(mdn, ds, 100)

    enet.eval(); dnet.eval(); snet.eval(); bcnet.eval(); mdn.eval()

    def efp_pol(p, g):
        return efp_sample(snet, enet, dnet, p, g, betas, abars, 5, 0.3)
    def bc_pol(p, g):
        with torch.no_grad():
            return bcnet(torch.FloatTensor(p).unsqueeze(0),
                        torch.FloatTensor(g).unsqueeze(0)).squeeze(0).numpy()
    def mdn_pol(p, g):
        with torch.no_grad():
            return mdn.sample(torch.FloatTensor(p).unsqueeze(0),
                             torch.FloatTensor(g).unsqueeze(0)).squeeze(0).numpy()

    pols = {'EFP (Ours)': efp_pol, 'BC (Mean)': bc_pol, 'MDN-BC': mdn_pol}

    print("\n[8] Evaluation...")
    np.random.seed(777)

    # In-distribution
    id_starts = [np.array([np.random.uniform(-2.5,-1.5), np.random.uniform(-2,2)]) for _ in range(25)]
    # OOD: starts near obstacles or at extreme positions
    ood_starts = [np.array([np.random.uniform(-0.8,-0.4), np.random.uniform(-2,2)]) for _ in range(15)]
    ood_starts += [np.array([np.random.uniform(-2.8,-2.5), np.random.uniform(-2.8,2.8)]) for _ in range(10)]

    results = {}
    for cond, starts, pert in [('In-Dist', id_starts, None),
                                ('OOD', ood_starts, None),
                                ('Perturb', id_starts[:15], 15)]:
        print(f"\n  === {cond} ===")
        results[cond] = {}
        for name, pol in pols.items():
            runs = [rollout(env, pol, s, 80, pert) for s in starts]
            sr = np.mean([r['success'] for r in runs])
            ad = np.mean([r['final_dist'] for r in runs])
            print(f"    {name:<14} success={sr:.0%}  dist={ad:.3f}")
            results[cond][name] = runs

    # ============ VISUALIZATION ============
    print("\n[9] Plotting...")

    fig = plt.figure(figsize=(22, 16))
    fig.suptitle('Entropic Flow Policy vs Baselines — Multi-Modal Navigation',
                 fontsize=15, fontweight='bold', y=0.99)

    mc = {'EFP (Ours)': '#1565C0', 'BC (Mean)': '#E53935', 'MDN-BC': '#43A047'}

    def draw_obs(ax):
        for xmin, xmax, ymin, ymax in env.obstacles:
            ax.add_patch(plt.Rectangle((xmin,ymin), xmax-xmin, ymax-ymin,
                         color='#424242', alpha=0.85, zorder=3))

    # Row 1: Trajectories per condition
    for ci, cond in enumerate(['In-Dist', 'OOD', 'Perturb']):
        ax = fig.add_subplot(3, 4, ci+1)
        draw_obs(ax)
        for name in pols:
            for r in results[cond][name][:10]:
                t = r['trajectory']
                ax.plot(t[:,0], t[:,1], color=mc[name], alpha=0.4, lw=0.9)
        ax.plot(*env.goal, 'r*', ms=14, zorder=5)
        ax.set_xlim(-3.1,3.1); ax.set_ylim(-3.1,3.1); ax.set_aspect('equal')
        ax.set_title(f'{cond} Trajectories', fontsize=11)
        ax.grid(True, alpha=0.2)
        if ci == 0:
            for n, c in mc.items(): ax.plot([],[],color=c,lw=2,label=n)
            ax.legend(fontsize=7, loc='lower left')

    # Expert demos
    ax = fig.add_subplot(3, 4, 4)
    draw_obs(ax)
    mode_colors = {'upper':'#1565C0','lower':'#E53935','middle':'#43A047'}
    for d in demos[:60]:
        t = np.array([s['state'] for s in d])
        ax.plot(t[:,0], t[:,1], color=mode_colors[d[0]['mode']], alpha=0.3, lw=0.7)
    ax.plot(*env.goal, 'r*', ms=14, zorder=5)
    ax.set_xlim(-3.1,3.1); ax.set_ylim(-3.1,3.1); ax.set_aspect('equal')
    ax.set_title('Expert Demos (3 modes)', fontsize=11)
    for m,c in mode_colors.items(): ax.plot([],[],color=c,label=m.capitalize())
    ax.legend(fontsize=7, loc='lower left'); ax.grid(True, alpha=0.2)

    # Row 2: Success rates
    for ci, cond in enumerate(['In-Dist', 'OOD', 'Perturb']):
        ax = fig.add_subplot(3, 4, 5+ci)
        names = list(pols.keys())
        rates = [np.mean([r['success'] for r in results[cond][n]]) for n in names]
        bars = ax.bar(range(len(names)), rates, color=[mc[n] for n in names],
                     width=0.6, edgecolor='black', lw=0.5)
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels([n.replace(' ','\n') for n in names], fontsize=8)
        ax.set_ylabel('Success Rate'); ax.set_title(f'{cond} Success', fontsize=11)
        ax.set_ylim(0, 1.15)
        for b, r in zip(bars, rates):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02,
                    f'{r:.0%}', ha='center', fontweight='bold', fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')

    # Avg distance grouped
    ax = fig.add_subplot(3, 4, 8)
    conds = ['In-Dist', 'OOD', 'Perturb']
    x = np.arange(len(conds)); w = 0.25
    for i, name in enumerate(pols):
        means = [np.mean([r['final_dist'] for r in results[c][name]]) for c in conds]
        ax.bar(x + i*w - w, means, w, label=name, color=mc[name], edgecolor='black', lw=0.5)
    ax.set_xticks(x); ax.set_xticklabels(conds)
    ax.set_ylabel('Avg Final Distance'); ax.set_title('Final Distance Comparison', fontsize=11)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3, axis='y')

    # Row 3: Entropy landscape, losses, entropy profiles, summary
    ax = fig.add_subplot(3, 4, 9)
    enet.eval()
    xs = np.linspace(-3, 3, 80); ys = np.linspace(-3, 3, 80)
    Hmap = np.zeros((80,80))
    gt = torch.FloatTensor(env.goal).unsqueeze(0)
    with torch.no_grad():
        for i, x in enumerate(xs):
            for j, y in enumerate(ys):
                Hmap[j,i] = enet(torch.FloatTensor([[x,y]]), gt).item()
    # Mask obstacles
    for i, x in enumerate(xs):
        for j, y in enumerate(ys):
            if env.in_obstacle(np.array([x,y])):
                Hmap[j,i] = np.nan
    im = ax.contourf(xs, ys, np.ma.masked_invalid(Hmap), levels=30, cmap='viridis_r')
    draw_obs(ax)
    ax.plot(*env.goal, 'r*', ms=14, zorder=5)
    ax.set_xlim(-3.1,3.1); ax.set_ylim(-3.1,3.1); ax.set_aspect('equal')
    ax.set_title('Learned Entropy Landscape', fontsize=11)
    plt.colorbar(im, ax=ax, label='H(s,g)', shrink=0.8)

    # Losses
    ax = fig.add_subplot(3, 4, 10)
    ax.plot(hl, label='Entropy', color='#1565C0', lw=1.5)
    ax.plot(sl, label='Score', color='#9C27B0', lw=1.5)
    ax.plot(bl, label='BC', color='#E53935', lw=1.5)
    ax.plot(ml, label='MDN', color='#43A047', lw=1.5)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('Training Losses', fontsize=11)
    ax.legend(fontsize=8); ax.set_yscale('log'); ax.grid(True, alpha=0.3)

    # Entropy profiles
    ax = fig.add_subplot(3, 4, 11)
    for r in results['In-Dist']['EFP (Ours)'][:15]:
        t = r['trajectory']
        hs = []
        with torch.no_grad():
            for p in t:
                hs.append(enet(torch.FloatTensor(p).unsqueeze(0), gt).item())
        c = '#1565C0' if r['success'] else '#FF9800'
        ax.plot(hs, color=c, alpha=0.6, lw=1.2)
    ax.set_xlabel('Timestep'); ax.set_ylabel('H(s,g)')
    ax.set_title('Entropy Along EFP Rollouts', fontsize=11)
    ax.plot([],[], color='#1565C0', label='Success')
    ax.plot([],[], color='#FF9800', label='Failure')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Summary
    ax = fig.add_subplot(3, 4, 12); ax.axis('off')
    txt = "RESULTS SUMMARY\n" + "="*40 + "\n\n"
    for cond in conds:
        txt += f"{cond}:\n"
        for name in pols:
            sr = np.mean([r['success'] for r in results[cond][name]])
            ad = np.mean([r['final_dist'] for r in results[cond][name]])
            txt += f"  {name:<14} {sr:5.0%}  d={ad:.3f}\n"
        txt += "\n"
    ax.text(0.05, 0.95, txt, transform=ax.transAxes, fontsize=10,
            va='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#FFF3E0', alpha=0.8))

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig('./efp_v2_results.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Plot saved!")


if __name__ == "__main__":
    main()


ENTROPIC FLOW POLICY — Multi-Modal Obstacle Navigation v2

[1] Generating demonstrations...
Generated 984 collision-free demonstrations
  Upper: 345, Lower: 339, Middle: 300

[2] Datasets...
  47232 transitions, 19680 pairs

[3] Training entropy estimator...
  Entropy 20/100: 0.0413
  Entropy 40/100: 0.0416
  Entropy 60/100: 0.0341
  Entropy 80/100: 0.0350
  Entropy 100/100: 0.0298

[4] Training dynamics...
  Dynamics 20/40: 0.0000
  Dynamics 40/40: 0.0000

[5] Training score network...
  Score 20/100: 0.3934
  Score 40/100: 0.3841
  Score 60/100: 0.3714
  Score 80/100: 0.3704
  Score 100/100: 0.3668

[6] Training BC...
  BC 20/60: 0.8309
  BC 40/60: 0.8196
  BC 60/60: 0.8145

[7] Training MDN-BC...
  MDN 20/100: 1.9827
  MDN 40/100: 1.9028
  MDN 60/100: 1.8818
  MDN 80/100: 1.8590
  MDN 100/100: 1.8290

[8] Evaluation...

  === In-Dist ===
    EFP (Ours)     success=76%  dist=0.348
    BC (Mean)      success=100%  dist=0.124
    MDN-BC         success=100%  dist=0.129

  === OOD ===
 